In [1]:
%%capture
%pip install -r requirements-common.txt

In [84]:
import json
import os

# Generic utils
from examples.utils import setup_notebook

# Check environment setup
if not setup_notebook(required_keys=["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]):
    raise OSError("Please set up required environment variables")

In [85]:
from urllib.parse import quote as url_quote

from requests import delete, get, post
from websockets import ConnectionClosed, connect

In [ ]:
response = post("http://localhost:8000/api/v2/agents", json={
    "mode": "conversational",
    "name": "Test Agent 2: examples/create-agent.ipynb",
    "version": "1.0.0",
    "description": (
        "This is a test agent created by the examples/create-agent.ipynb script."
    ),
    "runbook_raw_text": "# Objective\nYou are a helpful assistant.",
    "platform_configs": [{
        "kind": "bedrock",
        "region_name": os.environ["AWS_DEFAULT_REGION"],
        "aws_access_key_id": os.environ["AWS_ACCESS_KEY_ID"],
        "aws_secret_access_key": os.environ["AWS_SECRET_ACCESS_KEY"],
    }],
    "action_packages": [{
        "name": "browsing",
        "version": "1.0.0",
        "organization": "testorg",
        "url": "localhost:31415",
        "api_key": { "value": "some-api-key" },
        "allowed_actions": ["search", "browse"],
    }],
    "agent_architecture": {
        "name": "agent_architecture_default_v2",
        "version": "1.0.0",
    },
    "observability_configs": [],
    "question_groups": [],
    "extra": {
        "test": "test",
    },
})
print(response.json())


In [ ]:
response = get("http://localhost:8000/api/v2/agents")
print(json.dumps(response.json(), indent=2))

In [ ]:
name_url_encoded = url_quote("Test Agent 2: examples/create-agent.ipynb")
agent_by_name = get(f"http://localhost:8000/api/v2/agents/by-name?name={name_url_encoded}").json()
print(json.dumps(agent_by_name, indent=2))

# Get id to use in the next few examples
agent_id = agent_by_name["agent_id"]

In [ ]:
thread_response = post("http://localhost:8000/api/v2/threads", json={
    "agent_id": agent_id,
    "name": "Test Thread",
    "messages": [
        {
            "role": "user",
            "agent_metadata": {
                "test": "test",
            },
            "server_metadata": {},
            "content": [
                {
                    "kind": "text",
                    "text": "What can you do?",
                    "citations": [],
                },
            ],
        },
    ],
})
print(json.dumps(thread_response.json(), indent=2))

# Store thread_id for later use
thread_id = thread_response.json()["thread_id"]

In [ ]:
# Use the agent_id from your earlier create-agent example
uri = f"ws://localhost:8000/api/v2/runs/{agent_id}/stream"

async with connect(uri) as websocket:
    # Send initial payload
    await websocket.send(json.dumps({
        "agent_id": agent_id,
        "thread_id": thread_id,
    }))

    # Set up concurrent tasks for sending and receiving
    async def receive_messages():
        while True:
            try:
                message = await websocket.recv()

                # If the message['type'] == 'user_message_request', we need to produce
                # an input and send the reply back over the websocket.
                message_dict = json.loads(message)
                message_dict = (
                    message_dict['event'] if 'event' in message_dict
                    else message_dict
                )

                if 'type' not in message_dict:
                    message_dump = json.dumps(json.loads(message), indent=2)
                    print(f"Unknown message: {message_dump}")
                    continue

                if message_dict['type'] == 'user_message_request':
                    # Produce an input in the notebook.
                    users_input = input("Enter a reply:>")
                    # Send the reply back over the websocket.
                    await websocket.send(json.dumps(
                        {"type": "user_message_reply", "input": users_input},
                    ))
                elif (
                    message_dict['type'] == 'delta'
                    and message_dict['delta']['event_type'] == 'message_end'
                ):
                    for content in message_dict['delta']['data']['content']:
                        if content['kind'] == 'text':
                            print(content['text'])
                elif message_dict['type'] == 'error':
                    print(message_dict['message'])
                    print(message_dict['stack_trace'])
            except ConnectionClosed:
                print("Connection closed")
                break

    # Run both tasks concurrently
    await receive_messages()


In [ ]:
# Now list threads for the agent
specific_thread = get(f"http://localhost:8000/api/v2/threads?agent_id={agent_id}").json()
print(json.dumps(specific_thread, indent=2))

In [ ]:
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}").json()
print(json.dumps(specific_agent, indent=2))

In [ ]:
delete(f"http://localhost:8000/api/v2/agents/{agent_id}")

In [ ]:
# If we get the agent again, it should be gone
specific_agent = get(f"http://localhost:8000/api/v2/agents/{agent_id}")
print(json.dumps({
    "status_code": specific_agent.status_code,
    "response": specific_agent.json(),
}, indent=2))
